In [9]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import seaborn as sns



## Loading The Dataset

In [10]:
# Load the preprocessed DataFrame
df = pd.read_parquet('../data/interim/df_dtypes_fixed.parquet', engine='fastparquet')

## Define variables

In [11]:
X = df.drop(columns = ['is_fraud', 'fraud_type'])

We drop the **`fraud_type`** column, as it is not suitable for predicting fraudulent transactions and may introduce unnecessary complexity.


In [12]:
y = df['is_fraud']

## Train/Test Split

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

We use **stratification** because the target class is heavily imbalanced, and we want to preserve the distribution of the fraud class.

## Handling missing values

In [14]:
X_train.isna().sum()

transaction_id                              0
timestamp                                   2
sender_account                              0
receiver_account                            0
amount                                      0
transaction_type                            0
merchant_category                           0
location                                    0
device_used                                 0
time_since_last_transaction            717703
spending_deviation_score                    0
velocity_score                              0
geo_anomaly_score                           0
payment_channel                             0
ip_address                                  0
device_hash                                 0
time_since_last_transaction_missing         0
dtype: int64

We drop the two rows with missing **`timestamp`** values, as they represent a negligible portion of the dataset.

We will use **User-Level Median Imputation** to handle missing values in the **`time_since_last_transaction`** column. This strategy preserves each user's typical behavior and minimizes the impact on fraud detection signals.  

The median will be computed **only on the training dataset**, and the same transformation will be applied to the test dataset.

In [15]:
X_train = X_train.dropna(subset=['timestamp'])

In [16]:
X_test["timestamp"].isna().sum()

np.int64(1)

We apply the same step to the test dataset as well.

In [17]:
X_test = X_test.dropna(subset=['timestamp'])

In [18]:
# Compute per-sender median on training data
user_medians = X_train.groupby('sender_account')['time_since_last_transaction'].median()

# Map medians to the training data
X_train['time_since_last_transaction'] = X_train['time_since_last_transaction'].fillna(
    X_train['sender_account'].map(user_medians)
)


In [19]:
# Count remaining missing values
remaining_missing = X_train['time_since_last_transaction'].isna().sum()

# Calculate percentage
total_rows = len(X_train)
missing_percent = (remaining_missing / total_rows) * 100

print(f"Remaining missing values: {remaining_missing}")
print(f"Percentage of missing values: {missing_percent:.2f}%")

Remaining missing values: 28432
Percentage of missing values: 0.71%


Approximately 0.71% of the rows belong to users for whom all values in the time_since_last_transaction column are missing. These missing values are likely not due to data errors; instead, they correspond to new users who appear only once in the dataset.

Instead of leaving these values as NaN, we will set them to 0, which allows the model to handle them numerically while still capturing the notion of a “first transaction.”

We will also adjust the missing-value flag to capture only users with entirely missing values in this column before the imputation.

By setting time_since_last_transaction to 0 for such users, we retain behavioral information rather than averaging it away. This treatment preserves the signal that the account is new or low-activity, which can be informative for fraud detection.

In [20]:
X_train['time_since_last_transaction'] = X_train['time_since_last_transaction'].fillna(0)
X_test['time_since_last_transaction'] = X_test['time_since_last_transaction'].fillna(0)

In [21]:
# adjust the missing-value flag to capture only users with entirely missing history, not just individual transactions
X_train['time_since_last_transaction_missing'] = X_train['time_since_last_transaction'].isnull().astype(int)
X_test['time_since_last_transaction_missing'] = X_test['time_since_last_transaction'].isnull().astype(int)

Next, we apply the same logic to the test dataset, completing the missing-value imputation process.

In [22]:
# Map training medians to test data
X_test['time_since_last_transaction'] = X_test['time_since_last_transaction'].fillna(
    X_test['sender_account'].map(user_medians)
)

# Add missingness flag for test data
X_test['time_since_last_transaction_missing'] = X_test['time_since_last_transaction'].isna().astype(int)

## Fixing problemetic values

Next, we examine the features **`amount`** and **`time_since_last_transaction`** for invalid negative values, out-of-range values, extreme outliers, and unusual distributions, while excluding other numeric features as they were previously identified as well-engineered in the initial notebook.
In addition, we examine the **`timestamp`** column to determine the time range covered by the observations.

In [23]:
stats_table = X_train[['amount', 'time_since_last_transaction', 'timestamp']].describe(percentiles=[.01, .25, .5, .75, .9, .99])
stats_table

,amount,time_since_last_transaction,timestamp
count,3.999998e+06,3.999998e+06,3999998
mean,3.586044e+02,1.176600e+00,2023-07-02 23:30:32.073422
min,1.000000e-02,-8.777814e+03,2023-01-01 00:09:26.241974
1%,1.000000e-02,-7.410767e+03,2023-01-05 03:05:17.076935
25%,2.654000e+01,-2.186155e+03,2023-04-02 17:57:40.075809
50%,1.383300e+02,0.000000e+00,2023-07-03 00:24:31.386949
75%,5.032500e+02,2.189644e+03,2023-10-02 03:57:41.445373
90%,1.125950e+03,4.508445e+03,2023-11-25 22:07:35.249739
99%,1.874110e+03,7.410163e+03,2023-12-28 18:58:27.601108
max,3.520570e+03,8.757758e+03,2024-01-01 22:58:30.131850


In [24]:
X_train["spending_deviation_score"].describe()


count    3.999998e+06
mean    -3.683052e-04
std      1.000914e+00
min     -5.260000e+00
25%     -6.800000e-01
50%     -0.000000e+00
75%      6.700000e-01
max      5.020000e+00
Name: spending_deviation_score, dtype: float64

### amount
- Strong right skew (median ≈ 138 vs mean ≈ 359)
- Large but realistic outliers (max ≈ 3520)
- No obvious errors such as negative values → we keep it, but are going to transform it later (log)

### timestamp
- Covers full year (Jan 2023 → Jan 2024)
- Even distribution → suitable for time-based feature extraction







### time_since_last_transaction

An analysis of the descriptive statistics for **`time_since_last_transaction`** indicates that this feature is **standardized (scaled and centered)** rather than representing a raw temporal measurement or resulting from a sorting error.  

**Key evidence for standardization:**
- **Near-zero central tendency:** The mean (≈ 1.18) and median (≈ 1.81) are negligible compared to the standard deviation (≈ 3,353), indicating mean-centering.  
- **Symmetry:** The percentiles are nearly mirrored around zero (e.g., 25th percentile ≈ -2,207 vs. 75th percentile ≈ 2,210), a typical characteristic of standardized features.  
- **Exclusion of sorting errors:** While unsorted data can produce negative values, the consistent and balanced distribution strongly suggests a deliberate transformation (e.g., standard scaling).  

**Interpretation of values:**
- Negative values indicate transaction intervals shorter than the average.  
- Positive values indicate intervals longer than the average.  

Thus, negative values do not represent invalid or corrupted data, but rather reflect relative deviations from the mean.

## Feature Construction
### Time-Based Features

We apply each feature creation to the test dataset in the same manner, since no parameters need to be learned from it.

In [25]:
X_train['hour_of_day'] = X_train['timestamp'].dt.hour
X_test['hour_of_day'] = X_test['timestamp'].dt.hour

In [26]:
X_train["at_night"] = X_train['hour_of_day'].isin([21, 22, 23, 0, 1, 2, 3, 4, 5]).astype(int)
X_test["at_night"] = X_test['hour_of_day'].isin([21, 22, 23, 0, 1, 2, 3, 4, 5]).astype(int)

In [27]:
X_train['weekday'] = X_train['timestamp'].dt.weekday # Monday=0, Sunday=6
X_test['weekday'] = X_test['timestamp'].dt.weekday

In [28]:
X_train["on_weekend"] = X_train['weekday'].isin([5, 6]).astype(int)
X_test["on_weekend"] = X_test['weekday'].isin([5, 6]).astype(int)

### Transforming the Skewed Feature **`amount`**

In [29]:
X_train["log_amount"] = np.log(X_train["amount"])

Since transforming the **`amount`** feature using a log transformation does not require learning any parameters, we can apply the same transformation directly to the test dataset.

In [30]:
X_test["log_amount"] = np.log(X_test["amount"])

In [31]:
X_train["velocity_score"].describe()

count    3.999998e+06
mean     1.050116e+01
std      5.766916e+00
min      1.000000e+00
25%      5.000000e+00
50%      1.100000e+01
75%      1.600000e+01
max      2.000000e+01
Name: velocity_score, dtype: float64

### Interaction Features

The main idea behind creating interaction features is that fraud is rarely driven by a single variable; rather, it emerges from combinations of multiple signals.  

 The threshold defining "high" is learned from the training set only, so we simply adopt the same logic for the test set without recalculating it.

In [32]:
X_train["velocity_score"].describe()

count    3.999998e+06
mean     1.050116e+01
std      5.766916e+00
min      1.000000e+00
25%      5.000000e+00
50%      1.100000e+01
75%      1.600000e+01
max      2.000000e+01
Name: velocity_score, dtype: float64

In [33]:
# High amount + high velocity 
# We define "high" as above the , which corresponds to an amount greater than 503 and a velocity score greater than 1.6 in our training data.
X_train["high_amount_and_velocity"] = ((X_train["amount"] > 503) & (X_train["velocity_score"] > 1.6)).astype(int)
X_test["high_amount_and_velocity"] = ((X_test["amount"] > 503) & (X_test["velocity_score"] > 1.6)).astype(int)


In [34]:
X_train["geo_anomaly_score"].describe()

count    3.999998e+06
mean     5.000099e-01
std      2.886466e-01
min      0.000000e+00
25%      2.500000e-01
50%      5.000000e-01
75%      7.500000e-01
max      1.000000e+00
Name: geo_anomaly_score, dtype: float64

In [35]:
# High geo anomyly + high velocity
# We define "high" as above the 75th percentile for both features, which corresponds to a geo anomaly score greater than 0.75 and a velocity score greater than 1.6 in our training data.   

X_train["high_geo_anomaly_and_velocity"] = ((X_train["geo_anomaly_score"] > 0.75) & (X_train["velocity_score"] > 1.6)).astype(int)
X_test["high_geo_anomaly_and_velocity"] = ((X_test["geo_anomaly_score"] > 0.75) & (X_test["velocity_score"] > 1.6)).astype(int)

In [36]:
X_train["spending_deviation_score"].describe()

count    3.999998e+06
mean    -3.683052e-04
std      1.000914e+00
min     -5.260000e+00
25%     -6.800000e-01
50%     -0.000000e+00
75%      6.700000e-01
max      5.020000e+00
Name: spending_deviation_score, dtype: float64

In [37]:
# High deviation + high amount
# We define "high" as above the 75th percentile for both features, which corresponds to a spending deviation score greater than 0.67 and an amount greater than 503 in our training data.   

X_train["high_deviation_and_amount"] = ((X_train["spending_deviation_score"] > 0.67) & (X_train["amount"] > 503)).astype(int)
X_test["high_deviation_and_amount"] = ((X_test["spending_deviation_score"] > 0.67) & (X_test["amount"] > 503)).astype(int)

In [38]:
X_train.columns

Index(['transaction_id', 'timestamp', 'sender_account', 'receiver_account',
       'amount', 'transaction_type', 'merchant_category', 'location',
       'device_used', 'time_since_last_transaction',
       'spending_deviation_score', 'velocity_score', 'geo_anomaly_score',
       'payment_channel', 'ip_address', 'device_hash',
       'time_since_last_transaction_missing', 'hour_of_day', 'at_night',
       'weekday', 'on_weekend', 'log_amount', 'high_amount_and_velocity',
       'high_geo_anomaly_and_velocity', 'high_deviation_and_amount'],
      dtype='str')

In [39]:
X_train["time_since_last_transaction"].quantile(0.25)

np.float64(-2186.1551570493057)

In [40]:
# low time since last transaction + high geo anomaly
# We define "low" as below the 25th percentile for time since last transaction (<-2207) and "high" as above the 75th percentile for geo anomaly score (>0.75) in our training data.

X_train["low_time_and_high_geo_anomaly"] = ((X_train["time_since_last_transaction"] < -2207) & (X_train["geo_anomaly_score"] > 0.75)).astype(int)
X_test["low_time_and_high_geo_anomaly"] = ((X_test["time_since_last_transaction"] < -2207) & (X_test["geo_anomaly_score"] > 0.75)).astype(int)

In [41]:
# low time + high spending deviation
# We define "low" as below the 25th percentile for time since last transaction (<-2207) and "high" as above the 75th percentile for spending deviation score (>0.67) in our training data.  

X_train["low_time_and_high_deviation"] = ((X_train["time_since_last_transaction"] < -2207) & (X_train["spending_deviation_score"] > 0.67)).astype(int)
X_test["low_time_and_high_deviation"] = ((X_test["time_since_last_transaction"] < -2207) & (X_test["spending_deviation_score"] > 0.67)).astype(int)

### Binary/ flag Features

The idea here is to convert complex values into simple yes/no signals; for example, determining whether **`amount`** is extremely high.

We only consider values classified as "very high," defined as those above the 95th percentile.

In [42]:
X_train["amount"].quantile(0.95)

np.float64(1419.79)

In [43]:
X_train["very_high_amount"] = (X_train["amount"] > 1420).astype(int)
X_test["very_high_amount"] = (X_test["amount"] > 1420).astype(int)

In [44]:
X_train["geo_anomaly_score"].quantile(0.95)

np.float64(0.95)

In [45]:
X_train["very_high_geo_anomaly"] = (X_train["geo_anomaly_score"] > 0.95).astype(int)
X_test["very_high_geo_anomaly"] = (X_test["geo_anomaly_score"] > 0.95).astype(int)

In [46]:
X_train["velocity_score"].quantile(0.95)

np.float64(19.0)

In [47]:
X_train["very_high_velocity"] = (X_train["velocity_score"] > 19).astype(int)
X_test["very_high_velocity"] = (X_test["velocity_score"] > 19).astype(int)

In [48]:
X_train["time_since_last_transaction"].quantile(0.95) 


np.float64(5744.473680631)

In [49]:
# We will create a column long_inactivity which flags transactions from users who have been inactive for a long time, defined as having a time since last transaction greater than the 95th percentile  in our training data, which corresponds to a value greater than 5755.

X_train["long_inactivity"] = (X_train["time_since_last_transaction"] > 5755).astype(int)
X_test["long_inactivity"] = (X_test["time_since_last_transaction"] > 5755).astype(int)    

### Aggregation Features

The main idea is to leverage behavior across multiple transactions to create user-specific aggregated features that are not directly available in the raw data.

In [50]:
# 1. Compute averages ONLY on training data
sender_avg = X_train.groupby("sender_account")["amount"].mean()

# 2. Map to training data
X_train["avg_transaction_amount_per_sender"] = X_train["sender_account"].map(sender_avg)

# 3. Map to test data (IMPORTANT: use train stats!)
X_test["avg_transaction_amount_per_sender"] = X_test["sender_account"].map(sender_avg)
X_test["avg_transaction_amount_per_sender"] = X_test["avg_transaction_amount_per_sender"].fillna(0)

In [51]:
unique_receivers_per_sender = X_train.groupby("sender_account")["receiver_account"].nunique()

# Map to training data
X_train["unique_receivers_per_sender"] = X_train["sender_account"].map(unique_receivers_per_sender)

# Map to test data (IMPORTANT: use train stats!)
X_test["unique_receivers_per_sender"] = X_test["sender_account"].map(unique_receivers_per_sender)
X_test["unique_receivers_per_sender"] = X_test["unique_receivers_per_sender"].fillna(0)

In [52]:
accounts_per_device = X_train.groupby("device_hash")["sender_account"].nunique()

# Map to training data
X_train["accounts_per_device"] = X_train["device_hash"].map(accounts_per_device)

# Map to test data (IMPORTANT: use train stats!)
X_test["accounts_per_device"] = X_test["device_hash"].map(accounts_per_device)
X_test["accounts_per_device"] = X_test["accounts_per_device"].fillna(0)

In [53]:
transaction_counts_per_ip = X_train.groupby("ip_address").size()

# Map to training data
X_train["transactions_per_ip"] = X_train["ip_address"].map(transaction_counts_per_ip)

# Map to test data (IMPORTANT: use train stats!)
X_test["transactions_per_ip"] = X_test["ip_address"].map(transaction_counts_per_ip)
X_test["transactions_per_ip"] = X_test["transactions_per_ip"].fillna(0)

### Feature Selection / Cleanup 

In the following, we remove features that are redundant, uninformative, or have already been replaced by better-engineered features.

In [54]:
# Drop identifier and high-cardinality columns
# - transaction_id: pure identifier, no predictive value
# - timestamp: replaced by extracted time-based features
# - sender_account, receiver_account, device_hash, ip_address:
#   high-cardinality features whose information is captured via aggregation features
X_train = X_train.drop(["transaction_id", "timestamp", "sender_account", "receiver_account", "device_hash", "ip_address", "amount"], axis=1)
X_test = X_test.drop(["transaction_id", "timestamp", "sender_account", "receiver_account", "device_hash", "ip_address", "amount"], axis=1)

In [55]:
X_train.columns

Index(['transaction_type', 'merchant_category', 'location', 'device_used',
       'time_since_last_transaction', 'spending_deviation_score',
       'velocity_score', 'geo_anomaly_score', 'payment_channel',
       'time_since_last_transaction_missing', 'hour_of_day', 'at_night',
       'weekday', 'on_weekend', 'log_amount', 'high_amount_and_velocity',
       'high_geo_anomaly_and_velocity', 'high_deviation_and_amount',
       'low_time_and_high_geo_anomaly', 'low_time_and_high_deviation',
       'very_high_amount', 'very_high_geo_anomaly', 'very_high_velocity',
       'long_inactivity', 'avg_transaction_amount_per_sender',
       'unique_receivers_per_sender', 'accounts_per_device',
       'transactions_per_ip'],
      dtype='str')

In [56]:
X_train["location"].value_counts() 

location
Tokyo        500666
Singapore    500267
Berlin       500193
Sydney       500155
New York     500127
Dubai        499730
Toronto      499594
London       499266
Name: count, dtype: int64

We observe that the **`location`** column is not high-cardinality, but rather represents a clean categorical feature.

## Validation and Final Checks

In [57]:
X_train.head()

,transaction_type,merchant_category,location,device_used,time_since_last_transaction,spending_deviation_score,velocity_score,geo_anomaly_score,payment_channel,time_since_last_transaction_missing,...,low_time_and_high_geo_anomaly,low_time_and_high_deviation,very_high_amount,very_high_geo_anomaly,very_high_velocity,long_inactivity,avg_transaction_amount_per_sender,unique_receivers_per_sender,accounts_per_device,transactions_per_ip
1867199,deposit,retail,Dubai,atm,-2564.909222,-1.55,6,0.05,UPI,0,...,0,0,0,0,0,0,328.180000,8,1,1
2003209,payment,grocery,Sydney,web,-942.663664,0.69,2,0.59,wire_transfer,0,...,0,0,0,0,0,0,14.520000,3,1,1
1024599,deposit,grocery,London,mobile,-1491.579766,-0.21,2,0.70,UPI,0,...,0,0,0,0,0,0,227.133333,6,1,1
4901380,deposit,retail,Berlin,web,-5498.778833,-1.26,9,0.79,UPI,0,...,1,0,0,0,0,0,461.510000,2,1,1
1634088,deposit,travel,Sydney,atm,-3445.975296,-1.10,16,0.90,wire_transfer,0,...,1,0,0,0,0,0,262.118333,6,2,1


In [58]:
X_test.head()

,transaction_type,merchant_category,location,device_used,time_since_last_transaction,spending_deviation_score,velocity_score,geo_anomaly_score,payment_channel,time_since_last_transaction_missing,...,low_time_and_high_geo_anomaly,low_time_and_high_deviation,very_high_amount,very_high_geo_anomaly,very_high_velocity,long_inactivity,avg_transaction_amount_per_sender,unique_receivers_per_sender,accounts_per_device,transactions_per_ip
743476,transfer,retail,Singapore,pos,0.000000,-0.72,4,0.27,ACH,0,...,0,0,0,0,0,0,370.3900,2.0,0.0,0.0
1588704,transfer,other,Singapore,web,4984.368321,0.73,8,0.97,wire_transfer,0,...,0,0,0,1,0,0,195.3720,5.0,2.0,0.0
4415,transfer,online,Berlin,web,0.000000,-2.77,13,0.06,wire_transfer,0,...,0,0,0,0,0,0,9.8425,4.0,1.0,0.0
1173762,payment,other,Tokyo,atm,0.000000,-0.60,19,0.35,wire_transfer,0,...,0,0,0,0,0,0,48.0900,1.0,0.0,0.0
3426921,withdrawal,online,London,atm,-247.098577,0.26,10,0.67,UPI,0,...,0,0,0,0,0,0,211.5050,6.0,1.0,0.0


In [59]:
X_train.columns.equals(X_test.columns)

True

The final training and test datasets contain identical feature columns, indicating that feature engineering and categorical encoding were applied consistently across both datasets.

In [60]:
X_train.dtypes

transaction_type                        object
merchant_category                       object
location                                object
device_used                             object
time_since_last_transaction            float64
spending_deviation_score               float64
velocity_score                           int64
geo_anomaly_score                      float64
payment_channel                         object
time_since_last_transaction_missing      int64
hour_of_day                              int32
at_night                                 int64
weekday                                  int32
on_weekend                               int64
log_amount                             float64
high_amount_and_velocity                 int64
high_geo_anomaly_and_velocity            int64
high_deviation_and_amount                int64
low_time_and_high_geo_anomaly            int64
low_time_and_high_deviation              int64
very_high_amount                         int64
very_high_geo

Additionally, all features are in numerical format, ensuring compatibility with machine learning models.

In [61]:
X_train.isna().sum()

transaction_type                       0
merchant_category                      0
location                               0
device_used                            0
time_since_last_transaction            0
spending_deviation_score               0
velocity_score                         0
geo_anomaly_score                      0
payment_channel                        0
time_since_last_transaction_missing    0
hour_of_day                            0
at_night                               0
weekday                                0
on_weekend                             0
log_amount                             0
high_amount_and_velocity               0
high_geo_anomaly_and_velocity          0
high_deviation_and_amount              0
low_time_and_high_geo_anomaly          0
low_time_and_high_deviation            0
very_high_amount                       0
very_high_geo_anomaly                  0
very_high_velocity                     0
long_inactivity                        0
avg_transaction_

In [62]:
X_test.isna().sum()

transaction_type                       0
merchant_category                      0
location                               0
device_used                            0
time_since_last_transaction            0
spending_deviation_score               0
velocity_score                         0
geo_anomaly_score                      0
payment_channel                        0
time_since_last_transaction_missing    0
hour_of_day                            0
at_night                               0
weekday                                0
on_weekend                             0
log_amount                             0
high_amount_and_velocity               0
high_geo_anomaly_and_velocity          0
high_deviation_and_amount              0
low_time_and_high_geo_anomaly          0
low_time_and_high_deviation            0
very_high_amount                       0
very_high_geo_anomaly                  0
very_high_velocity                     0
long_inactivity                        0
avg_transaction_

There are no missing values in either the training or test dataset.

After completing all feature engineering steps, we ensure that the target variable **`y`** is properly aligned with the final feature sets **`X_train`** and **`X_test`**, particularly since some rows were removed due to missing timestamps.  



In [63]:
y_train = y_train.loc[X_train.index]
y_test = y_test.loc[X_test.index]

In [64]:
len(X_train)==len(y_train)


True

In [65]:
len(X_test)==len(y_test)

True

## Exporting Processed Datasets

Finally, we save the processed datasets to the `data/processed/` directory.  

Note that categorical encoding and feature scaling have been deferred, as these steps will be performed in the subsequent notebook, `03_modeling.ipynb`, specifically within the model pipelines.

In [70]:
X_train.to_parquet('../data/processed/X_train_processed.parquet', engine='fastparquet', compression='snappy')

In [71]:
X_test.to_parquet('../data/processed/X_test_processed.parquet', engine='fastparquet', compression='snappy')

In [75]:
y_train.to_frame(name='target').to_parquet('../data/processed/y_train.parquet', engine='fastparquet', compression='snappy')

In [76]:
y_test.to_frame(name='target').to_parquet('../data/processed/y_test.parquet', engine='fastparquet', compression='snappy')